# Create dim_date table

In [0]:
%sql
CREATE OR REPLACE TABLE bike_data.gold.dim_date (
    date_key                INT         NOT NULL,
    full_date               DATE        NOT NULL,
    year                    INT         NOT NULL,
    quarter                 INT         NOT NULL,
    month_number            INT         NOT NULL,
    month_name              STRING      NOT NULL,
    month_name_short        STRING      NOT NULL,
    day_of_month            INT         NOT NULL,
    day_of_year             INT         NOT NULL,
    day_of_week_number      INT         NOT NULL,
    day_of_week_name        STRING      NOT NULL,
    day_of_week_short       STRING      NOT NULL,
    is_weekend              BOOLEAN     NOT NULL,
    is_weekday              BOOLEAN     NOT NULL,
    is_leap_year            BOOLEAN     NOT NULL,
    week_of_year_iso        INT         NOT NULL,
    yyyy_mm                 STRING      NOT NULL,
    yyyy_qq                 STRING      NOT NULL,
    -- Removed DEFAULTs here
    is_current_year         BOOLEAN     NOT NULL,
    is_last_year            BOOLEAN     NOT NULL,
    is_current_quarter      BOOLEAN     NOT NULL,
    is_last_quarter         BOOLEAN     NOT NULL,
    is_current_month        BOOLEAN     NOT NULL,
    is_last_month           BOOLEAN     NOT NULL,
    fiscal_year             INT,
    fiscal_quarter          INT,
    fiscal_month            INT,
    load_date               DATE        NOT NULL,
    
    CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
)
USING DELTA;

#Populate dim_date table & write to gold

In [0]:
from pyspark.sql.functions import (
    col, lit, current_date, year, quarter, month, dayofmonth, dayofyear,
    date_format, when, pmod, dayofweek, concat, weekofyear, add_months
)

# 1. Generate sequence of dates (2000-01-01 to ~2035-12-31)
dates = spark.range(
    start=0,
    end=(365 * 36) + (36 // 4 + 1) + 100 
).select(
    (lit("2000-01-01").cast("date") + col("id").cast("int")).alias("full_date")
)

# 2. Define Reference Dates for Relative Flags
today = current_date()
last_month_date = add_months(today, -1)
last_quarter_date = add_months(today, -3)
last_year_date = add_months(today, -12)

# 3. Build the Dimension DataFrame
df = dates.select(
    # Surrogate key: yyyyMMdd
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month_number"),
    date_format("full_date", "MMMM").alias("month_name"),
    date_format("full_date", "MMM").alias("month_name_short"),
    dayofmonth("full_date").alias("day_of_month"),
    dayofyear("full_date").alias("day_of_year"),
    
    # ISO weekday: 1 = Monday ... 7 = Sunday
    (pmod(dayofweek("full_date") + 5, 7) + 1).alias("day_of_week_number"),
    date_format("full_date", "EEEE").alias("day_of_week_name"),
    date_format("full_date", "E").alias("day_of_week_short"),
    
    # Booleans
    when((pmod(dayofweek("full_date") + 5, 7) + 1).isin(6, 7), True).otherwise(False).alias("is_weekend"),
    when((pmod(dayofweek("full_date") + 5, 7) + 1) <= 5, True).otherwise(False).alias("is_weekday"),
    when(
        (year("full_date") % 4 == 0) & 
        ((year("full_date") % 100 != 0) | (year("full_date") % 400 == 0)), 
        True
    ).otherwise(False).alias("is_leap_year"),
    
    weekofyear("full_date").alias("week_of_year_iso"),
    date_format("full_date", "yyyy-MM").alias("yyyy_mm"),
    concat(date_format("full_date", "yyyy"), lit("-Q"), quarter("full_date")).alias("yyyy_qq"),
    
    # --- Relative Flags (Now calculated instead of just False) ---
    when(year("full_date") == year(today), True).otherwise(False).alias("is_current_year"),
    when(year("full_date") == year(last_year_date), True).otherwise(False).alias("is_last_year"),
    
    when((year("full_date") == year(today)) & (quarter("full_date") == quarter(today)), True).otherwise(False).alias("is_current_quarter"),
    when((year("full_date") == year(last_quarter_date)) & (quarter("full_date") == quarter(last_quarter_date)), True).otherwise(False).alias("is_last_quarter"),
    
    when((year("full_date") == year(today)) & (month("full_date") == month(today)), True).otherwise(False).alias("is_current_month"),
    when((year("full_date") == year(last_month_date)) & (month("full_date") == month(last_month_date)), True).otherwise(False).alias("is_last_month"),
    
    # Fiscal columns (Can be customized based on your company's fiscal start)
    lit(None).cast("int").alias("fiscal_year"),
    lit(None).cast("int").alias("fiscal_quarter"),
    lit(None).cast("int").alias("fiscal_month"),
    
    # The Load Date (Manually populating what was the DEFAULT)
    today.alias("load_date")
)

# 4. Write to Gold Layer
df.write.mode("overwrite").saveAsTable("bike_data.gold.dim_date")

In [0]:
df.display()

# Update current/last period flags

In [0]:
%sql
UPDATE bike_data.gold.dim_date
SET 
    is_current_year     = (year = YEAR(CURRENT_DATE())),
    is_last_year        = (year = YEAR(CURRENT_DATE()) - 1),
    
    is_current_quarter  = (year = YEAR(CURRENT_DATE()) AND quarter = QUARTER(CURRENT_DATE())),
    is_last_quarter     = (
        (year = YEAR(CURRENT_DATE()) AND quarter = QUARTER(CURRENT_DATE()) - 1)
        OR 
        (year = YEAR(CURRENT_DATE()) - 1 AND quarter = 4 AND QUARTER(CURRENT_DATE()) = 1)
    ),
    
    is_current_month    = (year = YEAR(CURRENT_DATE()) AND month_number = MONTH(CURRENT_DATE())),
    is_last_month       = (
        (year = YEAR(CURRENT_DATE()) AND month_number = MONTH(CURRENT_DATE()) - 1)
        OR 
        (year = YEAR(CURRENT_DATE()) - 1 AND month_number = 12 AND MONTH(CURRENT_DATE()) = 1)
    );